# Thai DSR — joint mapper + HiFi-GAN บน CUDA
เปิดแท็บเบราว์เซอร์ไว้และอย่าให้แล็ปท็อป sleep ระหว่างฝึก: Free/Pro ไม่มี background execution ที่รับประกันเมื่อปิดแท็บ การเชื่อมต่ออาจหลุดและ runtime อาจถูกยุติได้ ([Colab FAQ](https://research.google.com/colaboratory/faq.html)).

อัปโหลด `colab_transfer_bundle.zip` ไปที่ **My Drive/thai_dsr_colab/** แล้วเลือก **Runtime → Change runtime type → GPU** และรันเซลล์ตามลำดับ ยืนยันสิทธิ์ mount Drive เมื่อมีข้อความถาม

Full run: 2,000 steps; checkpoint ทุก 250 steps (ประมาณ 1.2 GiB ต่อ snapshot; เตรียม Drive ว่างประมาณ 12 GiB รวม bundle). ไม่มีการลบ checkpoint อัตโนมัติ หาก runtime ใหม่ ให้รันเซลล์ตั้งแต่ต้น; smoke จะข้ามเมื่อมี full checkpoint และ full cell จะ resume ล่าสุด ใช้ `MAX_STEPS` เป็นจำนวน steps รวม ไม่ใช่จำนวนเพิ่ม.

โค้ดฝึกเดิมไม่เปลี่ยน: mapper/generator/discriminators และ content encoder ใช้ CUDA; mel STFT/resampling บางส่วนยังอยู่ CPU ตาม implementation เดิม. ความเร็วจริงต้องวัดบน Colab.


In [ ]:
# 1. GPU
import torch
print('torch:', torch.__version__)
print('torch.cuda.is_available():', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
# 2. Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/thai_dsr_colab')  # EDIT if needed
BUNDLE = DRIVE_ROOT / 'colab_transfer_bundle.zip'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file(), f'Upload the zip to {BUNDLE}'


In [ ]:
# 3. Clone project (safe to rerun in the same runtime)
import os, subprocess, sys
REPO = Path('/content/thai-dsr')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/KankaveeRamsri/thai-dsr.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


In [ ]:
# 4. Dependencies: preserve Colab's matching CUDA torch/torchaudio builds.
# Use subprocesses so package changes do not leave stale imports in the trainer.
import importlib.metadata as metadata
constraints = Path('/content/thai_dsr_constraints.txt')
constraints.write_text(''.join(f'{name}=={metadata.version(name)}\n' for name in ('torch', 'torchaudio')))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
                'PyYAML>=6.0', 'transformers==5.13.1', '-c', str(constraints)], check=True)
subprocess.run([sys.executable, '-c',
    'import torch, torchaudio, yaml; from transformers import Wav2Vec2Model; '
    'assert torch.cuda.is_available(); print(torch.__version__, torchaudio.__version__)'], check=True)


In [ ]:
# 5. Extract at repo root and verify every bundled file (SHA-256).
import hashlib, json, shutil, zipfile
local_zip = Path('/content/colab_transfer_bundle.zip')
shutil.copyfile(BUNDLE, local_zip)
with zipfile.ZipFile(local_zip) as archive:
    for entry in archive.infolist():
        target = (REPO / entry.filename).resolve()
        assert target.is_relative_to(REPO.resolve()), entry.filename
    archive.extractall(REPO)
local_zip.unlink()
manifest = json.loads((REPO / 'scripts/colab_bundle_manifest.json').read_text())
for entry in manifest['files']:
    p = REPO / entry['path']
    assert p.stat().st_size == entry['bytes'], p
    digest = hashlib.sha256()
    with p.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    assert digest.hexdigest() == entry['sha256'], f'Checksum mismatch: {p}'
print('Verified', len(manifest['files']), 'files; HiFi-GAN revision:', manifest['hifigan_revision'])
# The trainer confines outputs to this tree. A symlink sends writes directly to Drive.
output_link = REPO / 'results/checkpoints/joint_finetune'
if output_link.is_symlink():
    assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
elif output_link.exists():
    raise RuntimeError(f'{output_link} already exists; preserve its contents before linking Drive.')
else:
    output_link.symlink_to(CHECKPOINT_ROOT, target_is_directory=True)
assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
# Download the public frozen encoder automatically (about 1.2 GiB weights).
# The trainer reads this cache by model name; no local Mac cache is required.
os.environ['HF_HOME'] = '/content/thai_dsr_hf'
os.environ.pop('HF_HUB_OFFLINE', None)
subprocess.run([sys.executable, '-c',
    'from huggingface_hub import snapshot_download; '
    'snapshot_download("airesearch/wav2vec2-large-xlsr-53-th", '
    'allow_patterns=["config.json", "preprocessor_config.json", "model.safetensors", "pytorch_model.bin"])'], check=True)
subprocess.run([sys.executable, '-m', 'src.training.joint_finetune', '--help'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_joint_finetune', '-v'], check=True)


In [ ]:
# 6. CUDA smoke: 50 steps, gradient checks + training and wall-clock throughput.
import time, uuid
from datetime import datetime
BASE = [sys.executable, '-u', '-m', 'src.training.joint_finetune',
        '--device', 'cuda', '--content-device', 'cuda',
        '--mapper-checkpoint', 'results/checkpoints/mapper_layer9.pt',
        '--manifest', 'data/manifest_w5.csv', '--splits', 'data/splits_w5.json',
        '--vocoder-init', 'thai', '--log-interval', '10']
def new_run(prefix):
    return output_link / (prefix + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])
def run_training(extra, output):
    command = BASE + ['--output-dir', str(output)] + extra
    print(' '.join(command), flush=True)
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, check=True)
    elapsed = time.perf_counter() - started
    summary = json.loads((output / 'summary.json').read_text())
    print(f"Training steps/sec: {1 / summary['mean_seconds']:.4f}")
    print(f"End-to-end steps/sec (startup, validation, saving): {summary['steps'] / elapsed:.4f}")
    print(f"Estimated 2000-step compute time: {2000 * summary['mean_seconds'] / 3600:.2f} hours + validation/saving")
    return summary
if list(CHECKPOINT_ROOT.glob('full_*/joint_*.pt')):
    print('Existing full checkpoint found; skip smoke and continue to resume cell.')
else:
    smoke_output = new_run('smoke')
    run_training(['--max-steps', '50', '--checkpoint-interval', '50',
                  '--validation-interval', '50', '--val-items', '3'], smoke_output)


## Full run / Resume
รันเซลล์ด้านล่างหลัง smoke ผ่าน (finite loss และ gradient checks). ค่าเริ่มต้น 2,000 steps และบันทึกทุก 250 steps ตรงลง `My Drive/thai_dsr_colab/checkpoints/full_.../` ผ่าน symlink รวม mapper, generator, MPD/MSD, optimizer และ RNG state.

เริ่ม full run ใหม่จากโมเดลต้นทางเมื่อยังไม่มี full checkpoint; ไม่ต่อจาก smoke. เมื่อกลับมาให้เลือก checkpoint ที่ step สูงสุดจาก full runs โดยอัตโนมัติ (ใช้เวลาแก้ไขไฟล์ตัดสินกรณี step เท่ากัน). `.tmp` จากการเขียนที่ถูกขัดจังหวะจะถูกข้าม; ถ้าไฟล์ `.pt` เสีย ให้ตั้ง `RESUME_PATH` เป็น checkpoint ก่อนหน้า. Resume ใช้โฟลเดอร์ใหม่เสมอ และยังต้องมี bundle ที่แตกไว้. Accelerator RNG ไม่ได้รับประกัน bitwise replay.


In [ ]:
# 7. FULL training; rerun this cell to resume after an interruption.
MAX_STEPS = 2000
CHECKPOINT_INTERVAL = 250
RESUME_PATH = None  # Optional explicit path to a previous joint_XXXXXXXX.pt
candidates = list(CHECKPOINT_ROOT.glob('full_*/joint_*.pt'))
latest = Path(RESUME_PATH) if RESUME_PATH else max(
    candidates, key=lambda p: (int(p.stem.split('_')[-1]), p.stat().st_mtime), default=None)
step = 0
if latest is not None:
    # Check that the snapshot is readable and complete before starting the trainer.
    state = torch.load(latest, map_location='cpu', weights_only=True, mmap=True)
    assert {'mapper', 'generator', 'mpd', 'msd', 'optim_g', 'optim_d', 'step',
            'sampling_rng', 'torch_rng'}.issubset(state), latest
    step = int(state['step'])
    del state
    print('Resume:', latest, 'step:', step)
if step >= MAX_STEPS:
    print('Target already reached. Increase MAX_STEPS to continue.')
else:
    output = new_run('full')
    extra = ['--max-steps', str(MAX_STEPS), '--checkpoint-interval', str(CHECKPOINT_INTERVAL),
             '--validation-interval', '250', '--val-items', '3']
    if latest is not None:
        extra += ['--resume', str(latest)]
    print('Checkpoints saved directly to:', output.resolve())
    run_training(extra, output)
